# Regression models — our day-ahead residual-load forecast vs. SMARD

Implements [`.claude/specs/05-regression-models.md`](../../.claude/specs/05-regression-models.md).

We forecast `residual_load` for every hour of a delivery day `DAY`, **issued at 18:00 on `DAY−1`**: the
same point in time as SMARD's public day-ahead forecast. Every model is scored against SMARD's
`fc_residual_load` and against a seasonal-naive floor, on **identical hours**.

**Our models are post-processors of SMARD's forecast.** They take SMARD's published component
forecasts (`fc_grid_load`, `fc_gen_wind_solar`) as inputs. If one wins, the honest claim is "we
reduce SMARD's error by X %", not "we forecast better than the TSOs".

## How to use this notebook

- Change a value in the configuration cells (§1.3), then re-run top to bottom. No later cell
  hardcodes a value the configuration holds.
- The interpretation is written **once**, in the closing section, for the default configuration.
  Tables and plots in between carry titles and units, no commentary.

## What this notebook produces

- day-ahead forecasts from seasonal naive `DAY−7`, `sarimax_fourier` and LightGBM direct / hybrid
  (XGBoost direct / hybrid when switched on), each under a **static** and a **rolling** split method
  on the same test year
- an empirical 95 % prediction interval per model, with its measured coverage
- one scoreboard (accuracy, extremes, intervals) including SMARD and seasonal naive
- two optional exports in `data/models/`, written only when `EXPORT_ENABLED` is on (default off)

## Conventions

- **Sign:** `error = forecast − actual`, as in spec 04. **Positive = over-forecast.**
- **Units:** hourly readings, forecasts and errors in `MWh`; capacity in `MW`; skill and coverage in
  `%`. The `MW` relabelling in `team-EDA.ipynb` does not apply here.
- **Durations, never row counts.** No literal calendar year or date appears in code.
- `time_series` holds exactly `SERIES + DERIVED`. Everything this notebook builds lives in separate
  frames.

## Not in this notebook

- Holt-Winters, seasonal ARIMA with a period `m`, MAPE, weather data, `fc_residual_load` as a feature
- risk flags on our forecast (parked [04.3](../../.claude/specs/04.3-risk-label-link.md)) and the
  remaining baselines of parked [04.1](../../.claude/specs/04.1-naive-baseline.md)
- significance tests, MLflow, any change to `modeling/`, reBAP

---

## 1 Setup and configuration

### 1.1 Shared setup (inherited)

Same setup as [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1, as reused by
[`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb) and
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb). Inherited,
not re-derived:

- the data-directory resolver (walks **upward** from the working directory)
- loading, renaming, the German-CSV float conversion and the dtype asserts
- `time_series`, `SERIES`, `DERIVED`, `YEARS`, `DAY_NAMES`, the season mapping
- `style_timeseries` (`ylabel` required)
- the duration pattern and the day-completeness rule from `risk-definition.ipynb` §3.1

Not needed here, so not repeated: `_complete_periods`, `period_mean`, `period_energy`,
`seasonal_plot` and the team-EDA colour configuration. Model colours live in the registry (§1.3).

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it with
`notebooks/API-connection.ipynb`.

In [ ]:
import itertools
import time
import warnings
from pathlib import Path

import holidays
import lightgbm as lgb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import xgboost as xgb
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import kpss

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(
    f"pandas {pd.__version__} · numpy {np.__version__} · statsmodels {statsmodels.__version__} · "
    f"lightgbm {lgb.__version__} · xgboost {xgb.__version__}"
)
print(f"Data directory: {DATA}")

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar"
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
    "cap_wind_off", "cap_wind_on", "cap_solar"
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.2 Resolution and durations

Every window, lag, refit interval, Fourier period, issue time and publication lag in this notebook is
a **duration**, converted to an observation count from the **measured** resolution by `n_obs`. A
switch to SMARD's quarter-hour data would change the counts, not the definitions.

At a sub-hourly resolution, models would be compared with each other at native resolution, and with
SMARD only after aggregating to hourly means, because `smard_forecast_errors_hourly.csv` holds
hourly errors. This notebook runs on the hourly `data/smard.csv` only.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

DAY_COMPLETENESS = 23 / 24   # accepts the spring-DST day, rejects materially short days


def n_obs(duration):
    """Observation count of `duration` at the measured resolution. Refuses a non-multiple."""
    count = duration / RESOLUTION
    if count != int(count):
        raise ValueError(f"{duration} is not a whole multiple of the resolution {RESOLUTION}")
    return int(count)


def describe(value):
    """Short readable form of a configuration value: hours below two days, whole days above."""
    if isinstance(value, pd.Timedelta):
        if value >= pd.Timedelta(days=2) and value == pd.Timedelta(days=value.days):
            return f"{value.days} days"
        return f"{value / pd.Timedelta(hours=1):g} h"
    if isinstance(value, pd.DateOffset):
        return ", ".join(f"{n} {unit.rstrip('s') if n == 1 else unit}" for unit, n in value.kwds.items())
    return str(value)


EXPECTED_OBS_PER_DAY = n_obs(pd.Timedelta(days=1))
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

print(f"resolution        : {describe(RESOLUTION)}  ->  {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"day completeness  : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations")

### 1.3 Configuration

Every value a later cell depends on is set here and nowhere else. Edit these the way you edit a
colour map, then re-run the notebook top to bottom. The defaults are the spec's.

| Cell | Holds |
|---|---|
| `DATA_INFO` | the forecast setting: issue time on `DAY−1`, the actuals publication lag, the capacity publication rule |
| `WINDOWS`, `TRAIN_TEST_SPLIT_METHOD` | test, validation and training window lengths, the refit interval, which split methods run |
| `MODELS` | the model registry: per model an on/off switch, family, architecture, fixed parameters, tuning grid, label and colour |
| `FEATURES` | one switch per booster feature group (**provisional**) and the durations the groups use |
| `INTERVAL`, `PLOT_MODEL`, `PLOT_SPLIT_METHOD`, `EXPORT_ENABLED` | interval level, forecast-plot selection, export toggle |

Windows and times are durations: `pd.Timedelta`, or `pd.DateOffset` where calendar months are meant.

In [ ]:
DATA_INFO = {
    "issue_days_before": pd.Timedelta(days=1),     # the forecast for DAY is issued on DAY−1 ...
    "issue_clock": pd.Timedelta(hours=18),         # ... at 18:00 local, once both SMARD components are out
    "actuals_lag": pd.Timedelta(hours=3),          # smard.de shows actuals ~2 h behind real time, +1 h margin
    "capacity_published_after": pd.DateOffset(years=1),  # year Y's cap_* value: 1 January of Y+1, 00:00
}

WINDOWS = {
    "test": pd.Timedelta(days=365),         # the last 365 complete delivery days
    "validation": pd.Timedelta(days=365),   # the 365 delivery days before the test window
    "train": pd.DateOffset(months=24),      # calendar months, not 730 days
    "refit_every": pd.Timedelta(days=30),   # rolling method only
}

TRAIN_TEST_SPLIT_METHOD = {"static": True, "rolling": True}

# Clock times implied by DATA_INFO, printed so the rule can be read without doing the arithmetic.
cutoff_clock = DATA_INFO["issue_clock"] - DATA_INFO["actuals_lag"]
last_actual_clock = cutoff_clock - RESOLUTION
first_capacity_use = pd.Timestamp(year=YEARS[0], month=1, day=1) + DATA_INFO["capacity_published_after"]

print("DATA_INFO")
print(
    f"  issue time            : {pd.Timestamp(0) + DATA_INFO['issue_clock']:%H:%M} on "
    f"DAY−{DATA_INFO['issue_days_before'].days}"
)
print(
    f"  actuals lag           : {describe(DATA_INFO['actuals_lag'])} -> availability cutoff "
    f"{pd.Timestamp(0) + cutoff_clock:%H:%M}, last usable actual ROW stamped "
    f"{pd.Timestamp(0) + last_actual_clock:%H:%M} on DAY−{DATA_INFO['issue_days_before'].days}"
)
print(
    f"  capacity publication  : {describe(DATA_INFO['capacity_published_after'])} after the start of "
    f"its year (the {YEARS[0]} value is usable from {first_capacity_use:%Y-%m-%d %H:%M})"
)
print("WINDOWS")
for name, value in WINDOWS.items():
    print(f"  {name:<22}: {describe(value)}")
print(f"TRAIN_TEST_SPLIT_METHOD : {TRAIN_TEST_SPLIT_METHOD}")

In [ ]:
SEED = 42  # every booster and every random draw in this notebook

# One grid and one parameter set per booster, shared by its direct and hybrid entries: a hybrid is
# tuned over exactly the same configurations as its direct variant.
LGBM_PARAMS = {"learning_rate": 0.05, "random_state": SEED, "verbose": -1}
LGBM_GRID = {"num_leaves": [31, 63], "n_estimators": [300, 800]}
XGB_PARAMS = {"learning_rate": 0.05, "random_state": SEED, "tree_method": "hist"}
XGB_GRID = {"max_depth": [4, 6], "n_estimators": [300, 800]}

MODELS = {
    "sarimax_fourier": {
        "enabled": True,
        "family": "sarimax",
        "architecture": None,
        "params": {
            "order": (1, 0, 1),                  # fixed, no order search; d = 0 (see the KPSS check)
            "trend": "c",                        # intercept of the regression
            "fourier": {pd.Timedelta(days=1): 4, pd.Timedelta(days=7): 3},  # period -> order K
            "exog": ["fc_grid_load", "fc_gen_wind_solar", "holiday"],
        },
        "grid": {},
        "label": "SARIMAX + Fourier",
        "color": "#D9A53A",
    },
    "lgbm_direct": {
        "enabled": True,
        "family": "lightgbm",
        "architecture": "direct",
        "params": LGBM_PARAMS,
        "grid": LGBM_GRID,
        "label": "LightGBM direct",
        "color": "#2C6EBA",
    },
    "lgbm_hybrid": {
        "enabled": True,
        "family": "lightgbm",
        "architecture": "hybrid",
        "params": LGBM_PARAMS,
        "grid": LGBM_GRID,
        "label": "LightGBM hybrid",
        "color": "#2F8F5B",
    },
    "xgb_direct": {
        "enabled": False,
        "family": "xgboost",
        "architecture": "direct",
        "params": XGB_PARAMS,
        "grid": XGB_GRID,
        "label": "XGBoost direct",
        "color": "#E95D0F",
    },
    "xgb_hybrid": {
        "enabled": False,
        "family": "xgboost",
        "architecture": "hybrid",
        "params": XGB_PARAMS,
        "grid": XGB_GRID,
        "label": "XGBoost hybrid",
        "color": "#B10F0F",
    },
}

# Rows outside the registry, with fixed colours. SMARD is drawn dashed.
FIXED = {
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A"},
    "seasonal_naive": {"label": "Seasonal naive (DAY−7)", "color": "#9098A2", "lag": pd.Timedelta(days=7)},
}

registry = pd.DataFrame(
    {
        key: {
            "enabled": m["enabled"],
            "family": m["family"],
            "architecture": m["architecture"] or "—",
            "grid configurations": int(np.prod([len(v) for v in m["grid"].values()])),
            "grid": m["grid"] or "—",
            "label": m["label"],
            "color": m["color"],
        }
        for key, m in MODELS.items()
    }
).T
print(f"MODELS: {sum(m['enabled'] for m in MODELS.values())} of {len(MODELS)} enabled, seed {SEED}")
display(registry)
print("fixed parameters")
for key, m in MODELS.items():
    params = {k: ({describe(p): o for p, o in v.items()} if k == "fourier" else v) for k, v in m["params"].items()}
    print(f"  {key:<16}: {params}")
print(f"outside the registry: {', '.join(FIXED)}")

In [ ]:
# Booster feature groups (spec Behaviour 18). PROVISIONAL: the team's feature-engineering work may
# replace them. They feed the direct boosters and the hybrids' stage 2 only. SARIMAX's inputs sit in
# its registry entry, and the hybrids' stage 1 always uses the two SMARD forecasts plus a trend.
FEATURES = {
    "calendar": True,             # local hour, day of week, month, is_weekend, holiday flag
    "smard_forecast": True,       # fc_grid_load, fc_gen_wind_solar for the target hour
    "lags": True,                 # residual_load at the same local hour on DAY−2 and DAY−7; last actual at the cutoff
    "recent_smard_error": True,   # mean err_grid_load and err_renewables over the window ending at the cutoff
    "capacity": True,             # cap_wind_off + cap_wind_on + cap_solar under the publication rule
}

FEATURE_WINDOWS = {
    "same_hour_lags": [pd.Timedelta(days=2), pd.Timedelta(days=7)],
    "recent_smard_error": pd.Timedelta(hours=24),
}

print("FEATURES (provisional)")
for group, enabled in FEATURES.items():
    print(f"  {group:<20}: {'on' if enabled else 'off'}")
print("FEATURE_WINDOWS")
for name, value in FEATURE_WINDOWS.items():
    shown = ", ".join(describe(v) for v in value) if isinstance(value, list) else describe(value)
    print(f"  {name:<20}: {shown}")

In [ ]:
INTERVAL = {"level": 0.95}      # empirical prediction interval, calibrated on the validation year

PLOT_MODEL = None               # None: the registry model with the lowest test MAE; or a key, e.g. "lgbm_hybrid"
PLOT_SPLIT_METHOD = "rolling"   # falls back to "static" when rolling is switched off

EXPORT_ENABLED = False          # True writes data/models/model_*.csv; the folder is not created here

print(f"INTERVAL          : {INTERVAL['level']:.0%} prediction interval")
print(f"PLOT_MODEL        : {PLOT_MODEL if PLOT_MODEL else 'None -> lowest test MAE among registry models'}")
print(f"PLOT_SPLIT_METHOD : {PLOT_SPLIT_METHOD}")
print(f"EXPORT_ENABLED    : {EXPORT_ENABLED}")

### 1.4 SMARD's hourly errors

`data/metrics/smard_forecast_errors_hourly.csv` is written by
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb) (spec 04). It
is used twice:

- to **re-score SMARD** on this notebook's common hours, instead of recomputing SMARD from
  `data/smard.csv`
- as the source of the `recent_smard_error` feature group (`err_grid_load`, `err_renewables`)

It stays in its own frame, `smard_errors`, and is never merged into `time_series`. The cell stops if
the file is missing, or if its timestamps differ from `time_series.index`: that would be an export
from another snapshot of `data/smard.csv`.

In [ ]:
SMARD_ERRORS = DATA_DIR / "metrics" / "smard_forecast_errors_hourly.csv"
SMARD_ERRORS_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SMARD_ERROR_COLUMNS = ["residual_load", "fc_residual_load", "err_residual_load", "err_grid_load", "err_renewables"]

if not SMARD_ERRORS.exists():
    raise FileNotFoundError(
        f"{SMARD_ERRORS} not found. data/metrics/ is gitignored, so the file is not in a fresh clone — "
        f"regenerate it by running {SMARD_ERRORS_SOURCE} top to bottom."
    )

smard_errors = pd.read_csv(SMARD_ERRORS)
smard_errors["timestamp"] = pd.to_datetime(smard_errors["timestamp"], format="%Y-%m-%d %H:%M:%S")
smard_errors = smard_errors.set_index("timestamp")

missing_columns = sorted(set(SMARD_ERROR_COLUMNS) - set(smard_errors.columns))
if missing_columns:
    raise ValueError(f"{SMARD_ERRORS.name} lacks {missing_columns} — re-run {SMARD_ERRORS_SOURCE}.")

if not smard_errors.index.equals(time_series.index):
    raise ValueError(
        f"{SMARD_ERRORS.name} does not match data/smard.csv: "
        f"{len(smard_errors.index.difference(time_series.index)):,} timestamps only in the errors file, "
        f"{len(time_series.index.difference(smard_errors.index)):,} only in smard.csv "
        f"(errors file {smard_errors.index.min()} -> {smard_errors.index.max()}). It is a stale export "
        f"from another snapshot — re-run {SMARD_ERRORS_SOURCE} top to bottom."
    )

print(f"smard_errors    : {len(smard_errors):,} rows, index identical to time_series")
print(f"range           : {smard_errors.index.min()}  ->  {smard_errors.index.max()}")
print(f"columns used    : {SMARD_ERROR_COLUMNS}")

### 1.5 Initial self-check

Structural checks on the loaded data and on the configuration, free of any hardcoded row count or
date. The closing self-check re-runs the data invariants and compares against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

# The configuration is internally consistent.
REGISTRY_KEYS = {"enabled", "family", "architecture", "params", "grid", "label", "color"}
for key, m in MODELS.items():
    assert REGISTRY_KEYS <= set(m), f"{key}: missing {REGISTRY_KEYS - set(m)}"
    assert m["family"] in {"sarimax", "lightgbm", "xgboost"}, (key, m["family"])
    if m["family"] == "sarimax":
        assert "fc_residual_load" not in m["params"]["exog"], f"{key}: fc_residual_load is never an input"
    else:
        assert m["architecture"] in {"direct", "hybrid"}, (key, m["architecture"])
    if m["architecture"] == "hybrid":
        direct = [d for d in MODELS.values() if d["family"] == m["family"] and d["architecture"] == "direct"]
        assert direct and m["grid"] == direct[0]["grid"], f"{key}: a hybrid uses its direct variant's grid"
assert not set(MODELS) & set(FIXED), "a registry key collides with a row outside the registry"
assert any(m["enabled"] for m in MODELS.values()), "switch on at least one registry model"
assert set(TRAIN_TEST_SPLIT_METHOD) == {"static", "rolling"}, TRAIN_TEST_SPLIT_METHOD
assert any(TRAIN_TEST_SPLIT_METHOD.values()), "switch on at least one split method"
assert PLOT_MODEL is None or MODELS.get(PLOT_MODEL, {}).get("enabled"), f"PLOT_MODEL {PLOT_MODEL!r} is not an enabled registry key"
assert PLOT_SPLIT_METHOD in TRAIN_TEST_SPLIT_METHOD, PLOT_SPLIT_METHOD
assert set(FEATURES) == {"calendar", "smard_forecast", "lags", "recent_smard_error", "capacity"}, FEATURES
assert 0 < INTERVAL["level"] < 1, INTERVAL
assert isinstance(EXPORT_ENABLED, bool)

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique, columns == SERIES + DERIVED")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")
print(f"  configuration consistent: {sum(m['enabled'] for m in MODELS.values())} registry models, "
      f"split methods {[k for k, v in TRAIN_TEST_SPLIT_METHOD.items() if v]}")